# Assignment 4. Color and Multi-Scale Representations

<span style="color:orange">Ground Rules for the Assignment: </span>

* <span style="color:lightblue"> You can only use basic functions (matrix operations, input/output image functions, plotters). Anything else, you need to code from scratch (histogram functions, inverting gamma functions, color matting, histogram equalization)</span>
* <span style="color:lightblue"> The code needs to be appropiately commented and should be reproducible; if we cannot re-generate your figures from your code, we will deduct points.</span>
* <span style="color:lightblue">The notebook report should be detailed and include partial and final solutions for each exercise. We grade solely the report; code without report will not be graded, so we encourage that you invest some time on it</span>
* <span style="color:lightblue">Interactive plots are welcome but most important results should be static and generated beforehand</span>
* <span style="color:lightblue">__Remember to remove all plots from the "coding" sections.__ Only the Report should output plots and/or images.</span>


<span style="color:orange">Submission Details</span>

Simply submit this Jupyter Notebook with the report inlined as described below. The notebook should be executed before submission. Name the file as 
```surname1_name1_surname2_name2_assignment3.ipynb```


In [ ]:
from copy import deepcopy

import cv2  #opencv python
import matplotlib.pyplot as plt
# Loading Libraries you will need for the assignment.- Install them in your environment if you haven't done so yet
import numpy as np
from PIL import Image
from matplotlib.axes import Axes
from sklearn.cluster import KMeans

# 0. Helpers

In [ ]:
def load_rgb_image(path: str, max_size: tuple[int, int] | None = None) -> Image.Image:
    """Load an image in RGB mode.

    A maximum size can optionally be provided to keep the notebook lightweight and
    reproducible when the original files are very large.
    """
    image = Image.open(path).convert("RGB")
    if max_size is not None:
        image.thumbnail(max_size, Image.Resampling.LANCZOS)
    return image


def image2array(image: Image.Image) -> np.ndarray:
    """Convert a PIL image to a float32 NumPy array in [0, 1]."""
    image = np.array(image)
    image = image.astype(np.float32) / 255.0
    return image


def array2image(image: np.ndarray) -> Image.Image:
    """Convert a float image in [0, 1] back to an 8-bit PIL image."""
    image = np.clip(image * 255, 0, 255).astype(np.uint8)
    return Image.fromarray(image)


def display(
        image: Image.Image | np.ndarray, gamma: float | None = None, title: str | None = None, ax: Axes | None = None
):
    """Display an image.

    gamma=None (or 0) means the image is already in display space.
    A positive gamma value is applied before display, which is useful for visualizing
    linear images.
    """
    if isinstance(image, Image.Image):
        image = image2array(image)

    if gamma not in (None, 0):
        image = np.clip(image, 0, 1) ** gamma

    cmap = None
    if image.ndim == 2:
        cmap = 'grey'

    if ax is not None:
        ax.imshow(np.clip(image, 0, 1), cmap=cmap)
        ax.set_title(title)
        ax.axis("off")
        return

    plt.imshow(np.clip(image, 0, 1), cmap=cmap)
    plt.title(title)
    plt.axis("off")

## 1 Color Palette Extraction from Images [8 Points]
Color palettes are a fundamental tool for both physical and digital artists. These sets of colors are a meaningful representation of all colors present in an image or piece of art, enabling the artist to tweak them for specific artistic effects. Purely decomposing an image into all of its colors can create "accurate" color palettes, but in natural images and paintings these would feature thousands of different colors and thus the palette becomes unmanageable. On the other extreme, a color palette solely composed of cyan, magenta and yellow (or red, green and blue in digital art) is a possible solution always, as these can create a wide range of visible colors, but they are not very helpful in describing the specific colors that best define a picture and thus using them to intuitively edit them or change their style is impossible. Best color palettes for digital image editing not only describe the colors that best define the style of a picture, but rather do so in a localized or semantic way. For example, the color of human skin and of a building painted pink could be rendered with similar colors, but editing them together to change the style of our picture would probably be undesired.

### 1.1 Linear RGB and sRGB Color Palettes
In this exercise, we ask you to implement a simple way of extracting a meaningful and interesting color palette from a single image. The end result will be the decomposition of the original image into a set of color layers that together add up to the initial image. We will first ask you to compute two different palettes, one starting from the image `queen.jpg` in linear RGB space, and the other in sRGB space. The process can be broadly described in 3 steps:
1. Loading Image in RGB space: Load the image and implement your own function to transform it to linear RGB (invert sRGB transform, see Lecture #8).
2. Color Clustering: We will now cluster those colors according to color similarity in RGB and sRGB spaces (this means that similar values will be clustered together, and its average will be a distinct palette color). To do this, you will use K-means clustering, so that they are divided in 7 clusters, selecting the centroid of each as representing color. You can use MATLAB’s function kmeans, but to do so you will first need to install MATLAB’s Statistics and Machine Learning Toolbox.
3. Image Decomposition into Base Color Layers: Now that you have your color palette, we will create 7 layers where we select only the pixels of the image whose colors we included in each individual cluster (see Figure 1). All other pixels will be set to black. The resulting layers will together add up to the initial image. Finally, translate each layer into HSL color space. This will enable intuitive editing of similar colors together, and applying changes to the image will limit to a simple addition of all layers. Additionally, draw a color strip with the centroids (average colors) of each filter/layer, which will be the representing colors of your palette (Figure 1).

Include a figure in your report with your resulting color layers from this step. Compare both linear and standard RGB palettes, arguing about quality.

In [ ]:
def srgb2rgb(image: Image.Image | np.ndarray) -> np.ndarray:
    if isinstance(image, Image.Image):
        image = image2array(image)

    image = np.clip(image, 0, 1)

    rgb = np.where(
        image <= 0.04045,
        image / 12.92,
        ((image + 0.055) / 1.055) ** 2.4
    )
    return rgb


def rgb2srgb(image: np.ndarray) -> np.ndarray:
    image = np.clip(image, 0, 1)

    srgb = np.where(
        image <= 0.0031308,
        12.92 * image,
        1.055 * (image ** (1 / 2.4)) - 0.055
    )
    return np.clip(srgb, 0, 1)


def kmeans_colors(
        image_rgb: np.ndarray,
        n_clusters: int = 7,
        seed: int = 42,
        return_estimator=False,
) -> tuple[np.ndarray, np.ndarray]:
    h, w, c = image_rgb.shape
    pixels = image_rgb.reshape(-1, 3).astype(np.float32)

    K = KMeans(n_clusters=n_clusters, random_state=seed)

    labels = K.fit_predict(pixels)
    centers = K.cluster_centers_

    labels_image = labels.reshape(h, w)

    if return_estimator:
        return labels_image, centers, K
    return labels_image, centers


def build_color_layers(
        original_display_rgb: np.ndarray,
        labels: np.ndarray,
) -> list[np.ndarray]:
    layers = []

    for cluster_id in np.unique(labels):
        mask = labels == cluster_id
        layer = np.zeros_like(original_display_rgb)
        layer[mask] = original_display_rgb[mask]
        layers.append(layer)

    return layers


def rgb_layers2hls(layers: list[np.ndarray]) -> list[np.ndarray]:
    hls_layers = []

    for layer in layers:
        layer_uint8 = np.clip(layer * 255, 0, 255).astype(np.uint8)
        hls_uint8 = cv2.cvtColor(layer_uint8, cv2.COLOR_RGB2HLS)
        hls_layers.append(hls_uint8.astype(np.float32) / 255.0)

    return hls_layers


def pallet_strip(
        centers_display_rgb: np.ndarray,
        height: int = 80,
        width_per_color: int = 120
) -> np.ndarray:
    """Create a horizontal color strip from RGB palette colors."""
    n_colors = len(centers_display_rgb)
    strip = np.zeros((height, width_per_color * n_colors, 3), dtype=np.float32)

    for i, color in enumerate(centers_display_rgb):
        x0 = i * width_per_color
        x1 = (i + 1) * width_per_color
        strip[:, x0:x1, :] = color

    return strip


def display_palette_result(
        original_rgb: np.ndarray,
        palette_strip: np.ndarray,
        layers: list[np.ndarray],
        title: str
):
    """Display original image, palette strip, and decomposed color layers."""
    n_layers = len(layers)

    fig = plt.figure(figsize=(18, 7))
    grid = fig.add_gridspec(
        2,
        n_layers + 1,
        height_ratios=[1, 2],
        width_ratios=[1] + [1] * n_layers
    )

    ax_original = fig.add_subplot(grid[:, 0])
    display(original_rgb, title="Original", ax=ax_original)

    ax_palette = fig.add_subplot(grid[0, 1:])
    ax_palette.imshow(np.clip(palette_strip, 0, 1))
    ax_palette.set_title(title)
    ax_palette.axis("off")

    for i, layer in enumerate(layers):
        ax = fig.add_subplot(grid[1, i + 1])
        display(layer, title=f"Layer {i + 1}", ax=ax)

    plt.tight_layout()
    plt.show()

In [ ]:
# Load image
queen_img = load_rgb_image("./queen.jpg")
queen_arr_srgb = image2array(queen_img)

n_clusters = 7

labels_srgb, centers_srgb = kmeans_colors(
    queen_arr_srgb,
    n_clusters=n_clusters,
)

layers_srgb = build_color_layers(
    queen_arr_srgb,
    labels_srgb,
)

hls_layers_srgb = rgb_layers2hls(layers_srgb)
palette_strip_srgb = pallet_strip(centers_srgb)

queen_arr_rgb = srgb2rgb(queen_arr_srgb)

labels_rgb, centers_rgb = kmeans_colors(
    queen_arr_rgb,
    n_clusters=n_clusters,
)

centers_rgb_display = rgb2srgb(centers_rgb)

layers_rgb = build_color_layers(
    queen_arr_srgb,
    labels_rgb,
)

hls_layers_rgb = rgb_layers2hls(layers_rgb)
palette_strip_rgb = pallet_strip(centers_rgb_display)

###  <span style="color:orange">  </span>
<span style="color:orange"> **Report.** _Report your results here._ </span>

In [ ]:
display_palette_result(
    queen_arr_srgb,
    palette_strip_srgb,
    layers_srgb,
    title="sRGB K-means palette"
)

display_palette_result(
    queen_arr_srgb,
    palette_strip_rgb,
    layers_rgb,
    title="RGB K-means palette"
)

### 1.2 CIELab Color Palette
You may have noticed that some clusters in your color palette are featuring colors that are quite different from each other, making editing a bit difficult. This is due sRGB not being a perceptually-uniform color space, which makes perceived and numerical differences between colors differ greatly. Perceptually uniform color spaces, such as CIELab, exploit computational models of human perception to create color spaces where geometric distances roughly correspond to similar distances in the perceptual space. This should ensure better color separation in your palette (Figure 1).

Edit your palette generation algorithm to use CIELab color space. You may use existing MATLAB/Python functions to translate your image from sRGB to CIELab. Compare both methods.

You can now edit the artistic look of your image intuitively by changing the individual color layers! Play around with them and generate some results of your own. You can try shifting the hue of specific layers, or plainly increasing lightness or saturation for specific effects. You should try it in at least 1 image of your own choice. Include the produced palette and filtered layers of the provided `queen.jpg` image as well.

![assignment4_fig1.png](attachment:assignment4_fig1.png)


**Bonus [1.5 Points]:** An important tool when editing the colors and overall feel of an image is white balance. As we saw in Assignment #1, obtaining a neutral, calibrated white balance is important for accurate photographic reproduction, but we can also alter it for artistic purposes. Recover your white balance code from Assignment #1 and try increasing or decreasing the temperature of your image after computing your color palette. Try then combining it with your palette-based color editing and other simple point operations we saw in Assignment #1 (contrast, brightness) to create your own tiny Lightroom! Showcase it with a picture of your choice.

In [ ]:
def rgb2lab(image: np.ndarray) -> np.ndarray:
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
    return lab


def lab2rgb(lab_image: np.ndarray) -> np.ndarray:
    lab_image = lab_image.astype(np.float32)
    rgb = cv2.cvtColor(lab_image, cv2.COLOR_LAB2RGB)
    return np.clip(rgb, 0, 1)


def edit_layer(
        layer_rgb: np.ndarray,
        hue_shift: float = 0.0,
        lightness_scale: float = 1.0,
        saturation_scale: float = 1.0
) -> np.ndarray:
    mask = layer_rgb.sum(axis=2) > 0

    layer_hls = cv2.cvtColor(layer_rgb, cv2.COLOR_RGB2HLS)

    hue = layer_hls[:, :, 0]
    lightness = layer_hls[:, :, 1]
    saturation = layer_hls[:, :, 2]

    hue[mask] = (hue[mask] + hue_shift) % 360
    lightness[mask] = np.clip(lightness[mask] * lightness_scale, 0, 1)
    saturation[mask] = np.clip(saturation[mask] * saturation_scale, 0, 1)

    edited_layer = np.stack([hue, lightness, saturation], axis=2)

    edited_layer_rgb = cv2.cvtColor(edited_layer, cv2.COLOR_HLS2RGB)
    edited_layer_rgb[~mask] = 0

    return edited_layer_rgb


def gray_world_white_balance(image_rgb: np.ndarray) -> np.ndarray:
    """White balance using the gray-world assumption."""
    image_rgb = np.clip(image_rgb, 0, 1).astype(np.float32)

    channel_mean = image_rgb.mean(axis=(0, 1))
    gray_mean = channel_mean.mean()

    gains = gray_mean / (channel_mean + 1e-6)

    corrected = image_rgb * gains
    return np.clip(corrected, 0, 1)

def edit_temperature(image_rgb, temperature: float = 0.0
) -> np.ndarray:
    """Simple artistic temperature shift.

    temperature > 0 makes image warmer.
    temperature < 0 makes image colder.
    """
    image_rgb = np.clip(image_rgb, 0, 1).astype(np.float32)

    result = image_rgb.copy()
    result[:, :, 0] *= 1.0 + temperature      # red
    result[:, :, 2] *= 1.0 - temperature      # blue

    return np.clip(result, 0, 1)

def display_editing_result(
        original_rgb: np.ndarray,
        edited_rgb: np.ndarray,
        original_layers: list[np.ndarray],
        edited_layers: list[np.ndarray],
        title: str
):
    """Display original image, edited image, and edited color layers."""
    n_layers = len(original_layers)

    fig = plt.figure(figsize=(18, 8))
    grid = fig.add_gridspec(
        2,
        n_layers + 2,
        width_ratios=[1, 1] + [1] * n_layers
    )

    ax_original = fig.add_subplot(grid[:, 0])
    display(original_rgb, title="Original", ax=ax_original)

    ax_edited = fig.add_subplot(grid[:, 1])
    display(edited_rgb, title="Edited image", ax=ax_edited)

    for i, layer in enumerate(edited_layers):
        ax = fig.add_subplot(grid[0, i + 2])
        display(original_layers[i], title=f"Original layer {i + 1}", ax=ax)

        ax = fig.add_subplot(grid[1, i + 2])
        display(layer, title=f"Edited layer {i + 1}", ax=ax)

    fig.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()


def display_lightroom_bonus(
        original_rgb: np.ndarray,
        palette_edited_rgb: np.ndarray,
        warm_rgb: np.ndarray,
        cold_rgb: np.ndarray,
        title: str
):
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))

    display(original_rgb, title="Original", ax=axes[0])
    display(palette_edited_rgb, title="Palette edited", ax=axes[1])
    display(warm_rgb, title="Warmer", ax=axes[2])
    display(cold_rgb, title="Colder", ax=axes[3])

    fig.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()

In [ ]:
# --- CIELab palette ---
queen_arr_lab = rgb2lab(queen_arr_srgb)

labels_lab, centers_lab = kmeans_colors(
    queen_arr_lab,
    n_clusters=n_clusters
)

layers_lab = build_color_layers(
    queen_arr_srgb,
    labels_lab,
)

centers_lab_rgb = lab2rgb(centers_lab.reshape(1, -1, 3)).reshape(-1, 3)

palette_strip_lab_rgb = pallet_strip(centers_lab_rgb)

edited_layers = deepcopy(layers_lab)

edited_layers[1] = edit_layer(edited_layers[1], hue_shift=12, saturation_scale=1.25)
edited_layers[3] = edit_layer(edited_layers[3], lightness_scale=0.75)

edited_queen_lab = np.sum(edited_layers, axis=0)

In [ ]:
# Bonus
palette_edited_queen = np.clip(edited_queen_lab, 0, 1)


balanced_queen = gray_world_white_balance(palette_edited_queen)

warm_queen = edit_temperature(balanced_queen, 0.12)
cold_queen = edit_temperature(balanced_queen, -0.12)

###  <span style="color:orange">  </span>
<span style="color:orange"> **Report.** _Report your results here._ </span>

In [ ]:
display_palette_result(
    queen_arr_srgb,
    palette_strip_srgb,
    layers_srgb,
    title="sRGB K-means palette"
)

display_palette_result(
    queen_arr_srgb,
    palette_strip_lab_rgb,
    layers_lab,
    title="CIELab K-means palette"
)

display_editing_result(
    queen_arr_srgb,
    edited_queen_lab,
    layers_lab,
    edited_layers,
    title="Artistic editing using CIELab palette layers"
)

In [ ]:
display_lightroom_bonus(
    queen_arr_srgb,
    palette_edited_queen,
    warm_queen,
    cold_queen,
    title="Bonus: white balance, temperature, brightness, and contrast"
)

## 2 Color Quantization and Lookup Tables (LUTs) [4 Points]
Quantization and color mapping through LUTs are techniques used in image processing to transform the color space of an image, either to reduce the amount of data required to represent the image, or to map colors from one space to another.

Quantization involves reducing the number of colors in an image by mapping similar colors to a smaller set of discrete values. This can be done by dividing the color space into a regular grid, and assigning each pixel to the closest grid point. This process is commonly used in image compression, as it reduces the amount of data required to represent the image, by storing per-pixel color IDs instead of exact colors. However, it can also cause loss of detail and introduce visual artifacts.

LUTs are used to map one set of values to another set of values. In image processing, LUTs are commonly used to map colors from one color space to another. For example, they can be used to convert an RGB image to a grayscale image, by mapping each pixel’s RGB values to a specific grayscale value. LUTs are also widely used in cinema production to apply color grading effects to an image or video, by mapping the original colors to new, adjusted colors, to achieve specific artistic effects.

**Task.** Implement a basic quantization method to reduce the size of the provided image by mapping its pixel values to a set of 32 colors. You may use any method to constrain this color space, except pre-defined MATLAB/Python solutions, of course. A few suggestions are, but not limited to: color clipping, thresholding, clustering, palette-based or euclidean distance-based nearest neighbors. Then, implement a look-up table that transforms your compressed RGB image into a grayscale image. You are also free to choose any method to map your 32 quantized colors to grey values, taking luminance, color difference or hue into account, for example. Try to maximize perceived quality, and make sure to explain in detail your approach for both quantization and LUT color mapping in you report. Show both your quantized color image and your resulting grayscale image after applying your LUT in the provided `queen.jpg` image.

![assignment4_fig2.png](attachment:assignment4_fig2.png)

**Bonus [1.5 Points]:** Make your color quantization pipeline perceptually-aware, by exploiting human color and luminance perception to quantize colors more aggressively or making quantization less obvious. Use the Weber-Fechner Law and perceptual color models like CAM or CIELab to improve quantization. Show this improvement comparatively.

In [ ]:
def quantize_with_noise(
        image_rgb: np.ndarray,
        palette_rgb: np.ndarray,
        noise_strength: float = 0.01,
        estimator=None,
        seed: int = 42
) -> tuple[np.ndarray, np.ndarray]:
    h, w, c = image_rgb.shape

    rng = np.random.default_rng(seed)

    noise = rng.normal(
        loc=0.0,
        scale=noise_strength,
        size=image_rgb.shape
    ).astype(np.float32)

    noisy_image = np.clip(image_rgb + noise, 0, 1)
    noisy_pixels = noisy_image.reshape(-1, 3)

    if estimator:
        labels = estimator.predict(noisy_pixels)
        labels = labels.reshape(h, w)
    else:
        raise ValueError("Estimator is not provided")

    quantized_rgb = palette_rgb[labels]

    return labels.astype(np.uint8), quantized_rgb


def image_metrics(
        original_rgb: np.ndarray,
        quantized_rgb: np.ndarray
) -> tuple[float, float]:
    mse = np.mean((original_rgb - quantized_rgb) ** 2)

    if mse == 0:
        psnr = float("inf")
    else:
        psnr = 10 * np.log10(1.0 / mse)

    return float(mse), float(psnr)


def display_quantization_result(
        original_rgb: np.ndarray,
        quantized_rgb: np.ndarray,
        grayscale_image: np.ndarray,
        palette_rgb: np.ndarray,
        title: str
):
    n_colors = len(palette_rgb)

    palette_strip = np.zeros((50, 24 * n_colors, 3), dtype=np.float32)

    for i, color in enumerate(palette_rgb):
        palette_strip[:, i * 24:(i + 1) * 24] = color

    fig = plt.figure(figsize=(16, 6))
    grid = fig.add_gridspec(2, 3, height_ratios=[4, 1])

    ax = fig.add_subplot(grid[0, 0])
    display(original_rgb, title="Original", ax=ax)

    ax = fig.add_subplot(grid[0, 1])
    display(quantized_rgb, title="32-color quantized", ax=ax)

    ax = fig.add_subplot(grid[0, 2])
    ax.imshow(np.clip(grayscale_image, 0, 1), cmap="gray", vmin=0, vmax=1)
    ax.set_title("Grayscale after LUT")
    ax.axis("off")

    ax = fig.add_subplot(grid[1, :])
    ax.imshow(np.clip(palette_strip, 0, 1))
    ax.set_title("32-color palette")
    ax.axis("off")

    fig.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()

In [ ]:
n_colors = 32


labels_rgb_32, palette_rgb_32, K = kmeans_colors(
    queen_arr_srgb,
    n_clusters=n_colors,
    return_estimator=True,
)


labels_noisy_32, quantized_noisy_32 = quantize_with_noise(
    queen_arr_srgb,
    palette_rgb_32,
    noise_strength=0.03,
    estimator=K,
)


luma_weights = np.array([0.2126, 0.7152, 0.0722], dtype=np.float32)

lut_rgb_32 = palette_rgb_32 @ luma_weights
lut_rgb_32 = np.clip(lut_rgb_32, 0, 1)

grayscale_noisy_32 = lut_rgb_32[labels_noisy_32]


compressed_rgb_32 = {
    "labels": labels_noisy_32,
    "palette": palette_rgb_32
}


mse_noisy_32, psnr_noisy_32 = image_metrics(
    queen_arr_srgb,
    quantized_noisy_32
)

###  <span style="color:orange">  </span>
<span style="color:orange"> **Report.** _Report your results here._ </span>

In [ ]:
display_quantization_result(
    queen_arr_srgb,
    quantized_noisy_32,
    grayscale_noisy_32,
    palette_rgb_32,
    title="RGB K-means quantization with white-noise dithering and luminance LUT"
)

print("--- Quantization quality ---")
print(f"Noisy RGB K-means | MSE: {mse_noisy_32:.6f} | PSNR: {psnr_noisy_32:.2f} dB")

## 3 Gaussian and Laplacian Pyramids [5 Points]
Multi-scale image representations (also called pyramids) transform an image into a set of images representing different frequency bands in the original image. While a Gaussian pyramid stores different low-pass versions of the original image, every level of Laplacian pyramid stores a specific range of spatial frequencies which correspond to band-passed versions of the original images. When all levels of Laplacian pyramid are added the result is the original image. This is not true for Gaussian pyramids.

**Task.** Implement a function to compute a Laplacian pyramid of an input image. The code should decompose the image into 4 levels. You are allowed to use MATLAB/Python functions for spatial filtering and resizing images. The most straightforward way to realize this task is to first build the Gaussian pyramid, and then subtract the consecutive levels to obtain the individual levels of the Laplacian pyramid (the procedure we described during the lecture). Compute the Laplacian pyramid of the two provided images (`sad.jpg` and `happy.jpg`) and show the different levels of your pyramid in your report.

In [ ]:
def build_gaussian_pyramid(
        image_rgb: np.ndarray,
        n_levels: int = 4
) -> list[np.ndarray]:
    gaussian_pyramid = [image_rgb]

    for i in range(1, n_levels):
        blurred = cv2.GaussianBlur(
            gaussian_pyramid[-1],
            ksize=(5, 5),
            sigmaX=1.0
        )

        h, w = blurred.shape[:2]

        downsampled = cv2.resize(
            blurred,
            (w // 2, h // 2),
            interpolation=cv2.INTER_LINEAR
        )

        gaussian_pyramid.append(downsampled)
    return gaussian_pyramid


def build_laplacian_pyramid(
        image_rgb: np.ndarray,
        n_levels: int = 4
) -> list[np.ndarray]:
    gaussian_pyramid = build_gaussian_pyramid(image_rgb, n_levels)

    laplacian_pyramid = []

    for i in range(n_levels - 1):
        current = gaussian_pyramid[i]
        next_level = gaussian_pyramid[i + 1]

        upsampled = cv2.resize(
            next_level,
            (current.shape[1], current.shape[0]),
            interpolation=cv2.INTER_LINEAR
        )

        laplacian = current - upsampled
        laplacian_pyramid.append(laplacian)

    laplacian_pyramid.append(gaussian_pyramid[-1])

    return laplacian_pyramid


def reconstruct_from_laplacian_pyramid(
        laplacian_pyramid: list[np.ndarray]
) -> np.ndarray:
    reconstructed = laplacian_pyramid[-1]

    for i in range(len(laplacian_pyramid) - 2, -1, -1):
        laplacian = laplacian_pyramid[i]

        upsampled = cv2.resize(
            reconstructed,
            (laplacian.shape[1], laplacian.shape[0]),
            interpolation=cv2.INTER_LINEAR
        )

        reconstructed = upsampled + laplacian

    return reconstructed


def display_laplacian_pyramid(
        original_rgb: np.ndarray,
        laplacian_pyramid: list[np.ndarray],
        title: str,
        contrast: float = 4.0
):
    n_levels = len(laplacian_pyramid)

    fig, axes = plt.subplots(1, n_levels + 1, figsize=(4 * (n_levels + 1), 4))

    display(original_rgb, title="Original", ax=axes[0])

    for i, level in enumerate(laplacian_pyramid):
        if i < n_levels - 1:
            visible = 0.5 + contrast * level
            visible = np.clip(visible, 0, 1)
            level_title = f"L{i}"
        else:
            visible = np.clip(level, 0, 1)
            level_title = f"L{i}: residual"

        display(visible, title=level_title, ax=axes[i + 1])

    fig.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()

In [ ]:
# Load images
sad_img = load_rgb_image("./sad.jpg")
happy_img = load_rgb_image("./happy.jpg")

sad_rgb = image2array(sad_img)
happy_rgb = image2array(happy_img)

n_pyramid_levels = 4


sad_laplacian_pyramid = build_laplacian_pyramid(
    sad_rgb,
    n_levels=n_pyramid_levels
)

happy_laplacian_pyramid = build_laplacian_pyramid(
    happy_rgb,
    n_levels=n_pyramid_levels
)


sad_reconstructed = reconstruct_from_laplacian_pyramid(sad_laplacian_pyramid)
diff = sad_rgb - sad_reconstructed

###  <span style="color:orange">  </span>
<span style="color:orange"> **Report.** _Report your results here._ </span>

In [ ]:
display_laplacian_pyramid(
    sad_rgb,
    sad_laplacian_pyramid,
    title="sad.jpg: Laplacian pyramid",
    contrast=4.0
)

display_laplacian_pyramid(
    happy_rgb,
    happy_laplacian_pyramid,
    title="happy.jpg: Laplacian pyramid",
    contrast=4.0
)

In [ ]:
print("mean signed error:", diff.mean())
print("MAE:", np.abs(diff).mean())
print("MSE:", np.mean(diff ** 2))
print("max abs error:", np.abs(diff).max())

## 4 Hybrid Images [3 Points]
A fun application of Gaussian and Laplacian pyramids is the creation of hybrid images. In essence, the human visual system is sensitive to high-frequency content of the image at close distances, but the further we move from the image our sensitivity and ability to resolve high-frequency content diminishes, and we better perceive low-spatial frequencies of the image. This can be exploited to create images which combine the high frequency component of one image with the low frequency component of another, resulting in an image that changes its appearance depending on the distance from which it is observed. A simple way of creating a hybrid image is to combine two different levels of Laplacian pyramids constructed for the input images. Another simple way is to filter one image with a low-pass filter and the other with complementary high-pass filter, and add the images together.

**Task.** Use one of the methods to compute hybrid images out of the two images `sad.jpg` and `happy.jpg` provided with the assignment. You should be able to get a similar effect to what you can see in Figure 3. If you use the first approach to computing hybrid images, play around with which levels to combine. When you use the second method, experiment with different cut-off frequencies for both filters. In both cases, see how the choices influence the quality of the results and the distances from which the images should be observed.

In the report provide the final image with information about the ideal viewing conditions:
- the size of the image on the screen
- the distance to see sad face
- the viewing distance to see happy face

![assignment4_fig3.png](attachment:assignment4_fig3.png)

In [ ]:
# Define your functions here .- you can include comments in your code explaining the steps of your algorithm

In [ ]:
# Execute your code here

###  <span style="color:orange">  </span>
<span style="color:orange"> **Report.** _Report your results here._ </span>

## 5 Bonus: Create your own Instagram Filter! [2 Points]
You may use now your newly acquired tools in image filtering and color manipulation to create your own Instagram filter! A possible approach can be to select a specific artist or photographer you like to emulate their style, or pick a known filter you want to reproduce. You can then employ specific color mappings or filtering methods from this and previous assignments to develop your own artistic rendition. You may also try your hand at color harmonization, as seen in Lecture #8. We only impose two rules:
1. Your filter should be plug-and-play, that is, we have to be able to use it on any image, regardless of colors, image size, etc.
2. It should be implemented as a single compact MATLAB/Python function so that we can test it out ourselves.

Otherwise, we give you full freedom to use any tool you want! Apply it to several of your own images, and showcase your solution with step-by-step partial result images illustrating the process from base image to your final result. Remember to include your source of inspiration as well in the report. Note: Copy pasting the grayscale effect developed in Exercise 2 will not render extra points. You can see an example filter developed by us in Figure 4.

![assignment4_fig4.png](attachment:assignment4_fig4.png)

In [ ]:
# Define your functions here .- you can include comments in your code explaining the steps of your algorithm

In [ ]:
# Execute your code here

###  <span style="color:orange">  </span>
<span style="color:orange"> **Report.** _Report your results here._ </span>